# 24. Korean Scent Domain Lexicon v0.1

Golden Set 200개의 실제 사용자 Query와 팀 원천 문서
`perfume_14families_korean_descriptors.md`를 대조해 서비스에서 재사용할 최소 한국어 향 표현
Lexicon을 만든다.

이 Notebook은 표현의 **표면형, 역할, 빈도, 원천 문서 근거**만 기록한다. Accord/Note와의 새
Semantic Mapping, Retrieval, Golden Set 수정, LLM/API 호출은 수행하지 않는다.

In [1]:
import hashlib
import json
import pathlib
import re
import unicodedata
from collections import defaultdict

import pandas as pd
from IPython.display import display

ROOT = pathlib.Path.cwd()
GOLD_PATH = ROOT / "evaluation_data" / "stage1" / "13_stage1_golden_set_v1_200.xlsx"
SOURCE_PATH = ROOT / "data" / "scent_knowledge" / "source" / "perfume_14families_korean_descriptors.md"
OUTPUT_PATH = ROOT / "data" / "scent_knowledge" / "korean_scent_lexicon_v0_1.csv"

GOLD_COLUMNS = [
    "query_id", "query_text", "gold_scent_preference", "gold_context",
    "gold_performance", "gold_avoid", "gold_additional_requirements",
]
ROLES = [
    "DIRECT_SCENT", "SENSORY", "SCENE", "SOURCE", "IMAGE", "CONTEXT",
    "PERFORMANCE", "AVOID", "REFERENCE", "OTHER",
]
ROLE_ORDER = {role: index for index, role in enumerate(ROLES)}

def file_sha256(path):
    digest = hashlib.sha256()
    with pathlib.Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

required = [GOLD_PATH, SOURCE_PATH]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"필수 입력이 없습니다: {missing}")

input_hashes_before = {path: file_sha256(path) for path in required}
print("Golden Set:", GOLD_PATH.relative_to(ROOT))
print("Source lexicon:", SOURCE_PATH.relative_to(ROOT))

Golden Set: evaluation_data\stage1\13_stage1_golden_set_v1_200.xlsx
Source lexicon: data\scent_knowledge\source\perfume_14families_korean_descriptors.md


## 1. 입력 고정과 재사용 범위

Golden Set의 사람이 작성한 `gold_avoid`와 `gold_additional_requirements`는 원문 Query에
포함된 표현을 그대로 재사용한다. 구조화 필드로만 남아 원문 표현이 빠진 직접 향 용어,
시즌/성별, 강도/지속력은 아래의 고정된 표면 패턴으로만 보완한다.

In [2]:
gold_raw_df = pd.read_excel(
    GOLD_PATH, sheet_name="Golden Set", header=4, dtype=str, keep_default_na=False,
)
if gold_raw_df.columns.tolist() != GOLD_COLUMNS or len(gold_raw_df) != 200:
    raise ValueError(
        f"Golden Set schema/row 오류: columns={gold_raw_df.columns.tolist()}, rows={len(gold_raw_df)}"
    )
if gold_raw_df["query_id"].duplicated().any() or gold_raw_df["query_text"].duplicated().any():
    raise ValueError("Golden Set query_id/query_text 중복")

gold_df = gold_raw_df.copy()
for column in [
    "gold_scent_preference", "gold_context", "gold_performance",
    "gold_avoid", "gold_additional_requirements",
]:
    gold_df[column] = gold_df[column].map(json.loads)

source_text = SOURCE_PATH.read_text(encoding="utf-8")
source_lines = source_text.splitlines()

annotated_items = []
for row in gold_df.itertuples(index=False):
    for field in ["gold_avoid", "gold_additional_requirements"]:
        for item in getattr(row, field):
            expression = item.strip()
            if not expression or expression not in row.query_text:
                raise ValueError(f"원문에 없는 annotation: {row.query_id} / {field} / {expression!r}")
            annotated_items.append((row.query_id, field, expression))

print("Golden Set rows:", len(gold_df))
print("Reused exact annotation items:", len(annotated_items))
print("Golden SHA256:", input_hashes_before[GOLD_PATH])
print("Source SHA256:", input_hashes_before[SOURCE_PATH])

Golden Set rows: 200
Reused exact annotation items: 312
Golden SHA256: a56613ad4bb082d2a05408f0a888727945cbc924a33ca064140a5c565589c3df
Source SHA256: 356762eb194bd8d39fc1753f7518682e2a702e788270592a89c0a4fb71bb8cf1


## 2. 원천 Lexicon의 명시 표현 인덱스

Family 정보는 14개 Family 절의 핵심어·감각어·생활 비유·부정 표현처럼 목록으로 명시된
항목에서만 가져온다. 문서의 검색 동의어와 검색자 어휘 절은 Family를 추정하지 않고 근거
표현으로만 사용한다.

- `MATCHED`: 원천 문서의 명시 표현 또는 명시된 표기 변형과 공백·문장부호를 제외하고 일치
- `RELATED`: 원천 명시 표현과 부분 문자열 또는 보수적인 굴절 어간이 일치
- `NOT_FOUND`: 위의 문자열 근거가 없음

`RELATED`는 의미가 같다는 판정이 아니며, Accord/Note 관계로 사용하지 않는다.

In [3]:
FAMILY_NAMES = [
    "Floral", "Soft Floral", "Floral Amber", "Soft Amber", "Amber",
    "Woody Amber", "Woods", "Mossy Woods", "Dry Woods",
    "Aromatic Fougère", "Citrus", "Water", "Green", "Fruity",
]
FAMILY_ALIASES = {
    "플로럴": "Floral", "소프트 플로럴": "Soft Floral",
    "플로럴 앰버": "Floral Amber", "소프트 앰버": "Soft Amber",
    "앰버": "Amber", "우디 앰버": "Woody Amber", "우디": "Woods",
    "우즈": "Woods", "모시 우즈": "Mossy Woods", "시프레": "Mossy Woods",
    "드라이 우드": "Dry Woods", "드라이 우즈": "Dry Woods",
    "푸제르": "Aromatic Fougère", "아로마틱 푸제르": "Aromatic Fougère",
    "시트러스": "Citrus", "워터": "Water", "아쿠아틱": "Water",
    "그린": "Green", "프루티": "Fruity",
}

def clean_markdown_term(value):
    value = re.sub(r"\[[Ss]\d+\]", "", value)
    value = re.sub(r"\([^)]*(?:→|[Ss]\d+)[^)]*\)", "", value)
    value = re.sub(r"[*_`]+", "", value)
    return re.sub(r"\s+", " ", value).strip(" \t-—:;,.\"'")

SOURCE_TERM_EXCLUSIONS = {
    "스러운", "같은", "좋은", "낮은", "높은", "직접", "중요", "완료", "유지", "추가",
}

def valid_source_term(term):
    return (
        1 < len(term) <= 80 and bool(re.search(r"[가-힣]", term))
        and not term.startswith(("근거", "검증", "출처", "주의"))
        and "http" not in term
        and term not in SOURCE_TERM_EXCLUSIONS
    )

source_rows = []
current_family = ""
current_subsection = ""
allowed_heading_tokens = (
    "핵심 이미지", "형용사", "감각어", "표현", "한국 후기식 생활 비유",
    "호불호·부정", "쪽", "계열", "질감", "당도", "무게", "습도",
)

for line in source_lines:
    family_match = re.match(r"^#\s+\d+\.\s+(.+?)\s+—", line)
    if family_match:
        candidate = family_match.group(1).strip()
        current_family = candidate if candidate in FAMILY_NAMES else ""
        current_subsection = ""
        continue
    if line.startswith("# ") and not family_match:
        current_family = ""
        current_subsection = ""
    if current_family and re.match(r"^#{2,3}\s+", line):
        current_subsection = clean_markdown_term(re.sub(r"^#{2,3}\s+", "", line))
        continue
    if not current_family or not any(token in current_subsection for token in allowed_heading_tokens):
        continue
    if "후기 작성용 조합" in current_subsection:
        continue

    candidates = []
    stripped = line.strip()
    if current_subsection == "핵심 이미지" and "**" in stripped:
        for part in re.findall(r"\*\*(.+?)\*\*", stripped):
            candidates.extend(re.split(r"\s*[/+]\s*", part))
    elif stripped.startswith("- "):
        candidates.append(stripped[2:])
    elif stripped and not stripped.startswith((">", "|", "**근거", "**검증")) and "," in stripped:
        candidates.extend(stripped.split(","))

    for candidate in candidates:
        term = clean_markdown_term(candidate)
        if valid_source_term(term):
            source_rows.append(
                {"source_term": term, "source_section": current_subsection, "source_family": current_family}
            )

# 검색 동의어·검색자 어휘 등에서 명시적으로 강조된 표현은 Family 추정 없이 추가한다.
for line_number, line in enumerate(source_lines, start=1):
    if not (1100 <= line_number <= 1692):
        continue
    for candidate in re.findall(r"`([^`]+)`|\*\*([^*]+)\*\*", line):
        term = clean_markdown_term(next(value for value in candidate if value))
        if valid_source_term(term):
            source_rows.append(
                {"source_term": term, "source_section": f"원천 문서 line {line_number}", "source_family": ""}
            )

    if line.strip().startswith("|"):
        cells = [clean_markdown_term(cell) for cell in line.strip().strip("|").split("|")]
        relevant_cells = []
        if 1513 <= line_number <= 1549:
            relevant_cells = cells[:2]
        elif 1608 <= line_number <= 1628 and len(cells) >= 2:
            relevant_cells = [cells[1]]
        for cell in relevant_cells:
            if not cell or set(cell) <= {"-", ":"} or cell in {"표제어", "변형 표기", "대상", "비유 표현들", "예", "실제 표현"}:
                continue
            for candidate in re.split(r"\s*[,/]\s*", cell):
                term = clean_markdown_term(candidate)
                if valid_source_term(term):
                    source_rows.append({
                        "source_term": term,
                        "source_section": f"원천 문서 table line {line_number}",
                        "source_family": "",
                    })

for alias, family in FAMILY_ALIASES.items():
    source_rows.append(
        {"source_term": alias, "source_section": "0. 분류 기준 / Family 명칭", "source_family": family}
    )

source_terms_df = pd.DataFrame(source_rows).drop_duplicates().reset_index(drop=True)
EXPLICIT_VARIANTS = {
    "쓸쿰한": "쿰쿰한", "퀴퀴한": "쿰쿰한", "쾨쾨한": "쿰쿰한",
    "상콤한": "상큼한", "보송한": "뽀송한", "뽀송뽀송한": "뽀송한",
    "폭닥폭닥한": "폭닥한", "폭신한": "폭닥한", "매쾌한": "매캐한",
    "화플": "화이트 플로럴", "그리너리함": "그린 노트의 느낌",
    "바닐라리함": "바닐라 같은 성질",
}
for variant, representative in EXPLICIT_VARIANTS.items():
    matched = source_terms_df.loc[source_terms_df["source_term"].eq(representative)]
    families = sorted(value for value in matched["source_family"].unique() if value)
    source_rows.append({
        "source_term": variant,
        "source_section": "20-B. 검색 동의어·변형 표기 사전",
        "source_family": "|".join(families),
    })

source_terms_df = pd.DataFrame(source_rows).drop_duplicates().reset_index(drop=True)
print("Parsed explicit source terms:", len(source_terms_df))
print("Family-linked source terms:", int(source_terms_df["source_family"].ne("").sum()))

Parsed explicit source terms: 1714
Family-linked source terms: 1080


## 3. 사용자 표현 추출과 역할 분류

한 표현에는 Pilot v0.1에서 가장 직접적인 역할 하나만 부여한다. 부정 조건은 내용이 Note나
감각어여도 `AVOID`가 우선한다. 장면과 실제 사용 상황은 각각 `SCENE`, `CONTEXT`로 구분하고,
제품/브랜드를 기준점으로 삼는 표현은 `REFERENCE`로 둔다.

In [4]:
def has_any(text, patterns):
    return any(re.search(pattern, text, flags=re.IGNORECASE) for pattern in patterns)

NEGATIVE_PATTERNS = [
    r"않았으면", r"안 아팠", r"안 어지", r"안 겹", r"안 이상", r"안나는",
    r"답답하지", r"날카롭지", r"강하진", r"보다는", r"인위적이면 싫",
]
SCENE_PATTERNS = [
    r"비가 온 뒤", r"^숲$", r"호수에 떠있는 연꽃", r"바다에 누워", r"하늘을 올려다",
    r"휴양지", r"햇살 내리쬐는 일본 여름", r"세탁한 포근한 이불", r"호텔 로비",
    r"새벽 바닷가", r"비 오는 숲", r"햇빛 받은 흰 셔츠", r"호텔 침구",
    r"목조 도서관", r"잔디밭에 누워", r"헌책방", r"포근한 니트",
    r"따뜻한 차 한 잔", r"창밖에 비 오는", r"이슬을 머금은 장미",
    r"물 대신 과즙을 마시고 자란 꽃", r"장마철", r"마사지숍",
]
REFERENCE_PATTERNS = [
    r"이솝", r"조말론", r"메종 마르지엘라", r"템버린즈", r"탬버린즈", r"양키캔들",
    r"샤넬", r"딥티크", r"오르페옹", r"블루 드 샤넬", r"리한나가 쓰는 향수",
    r"산타마리아노벨라", r"크리드", r"단종된 향수랑 비슷", r"레이어링", r"믹스해서",
]
DIRECT_PATTERNS = [
    r"계열", r"노트", r"^인센스$", r"^아이리스$", r"^패출리$", r"^파우더리$",
    r"^머스크$", r"화이트 머스크", r"오렌지 블로썸", r"^베티버$",
]
SOURCE_PATTERNS = [
    r"비누\s*향", r"섬유유연제", r"살냄새", r"세제", r"꽃향", r"꿀( 같은)? 향",
    r"과일냄새", r"홍차향", r"베이비 파우더", r"향신료", r"바디스프레이",
    r"딸기 향기", r"장미 꽃 향기", r"소금 냄새", r"흙냄새", r"나무 냄새",
]
PERFORMANCE_PATTERNS = [
    r"은은", r"잔향", r"오래가", r"지속력", r"강하게", r"너무 세", r"진한 향",
    r"뿌린 티", r"멀리까지 퍼", r"강도", r"머리.*아프", r"코 아프", r"어지러",
    r"부담 없", r"부담없이", r"민폐", r"가벼운",
]
CONTEXT_PATTERNS = [
    r"\d+대", r"\d+살", r"금융맨", r"선물", r"회사", r"매일", r"입문", r"면접",
    r"출근", r"헬스장", r"공식석상", r"소개팅", r"데이트", r"사계절", r"대학교",
    r"데일리", r"운동", r"사무실", r"강의실", r"직업", r"정장차림", r"옷에 향수",
    r"남자친구", r"여자친구", r"여친", r"남친", r"아빠", r"오늘",
]
IMAGE_PATTERNS = [
    r"공주", r"어른스러운", r"좋은 인상", r"조용", r"차분", r"세련", r"섹시",
    r"귀엽", r"릴랙싱", r"고급스럽", r"시크", r"카리스마", r"딱딱", r"성숙",
    r"젊은 느낌", r"담백", r"개성", r"서늘한 여자", r"엣지", r"중성적",
    r"여성스러운", r"자연스러운",
]
SENSORY_PATTERNS = [
    r"깔끔", r"깨끗", r"달콤", r"달달", r"포근", r"폭닥", r"맑", r"산뜻",
    r"쨍", r"차가", r"화한", r"부드럽", r"부드러", r"가볍", r"편안", r"시원",
    r"청량", r"상쾌", r"풍성", r"따뜻", r"싱싱", r"무거운 향", r"톡 쏘", r"과즙",
]

ROLE_OVERRIDES = {
    "부드러운": "SENSORY", "가볍고": "SENSORY", "좋은 냄새": "SENSORY",
    "청량": "SENSORY", "고급스러운": "IMAGE", "겨울 느낌": "IMAGE",
    "향이 너무 좋아서 사람들이 다 돌아볼 것 같은": "IMAGE",
    "과일 향수": "DIRECT_SCENT", "완전히 다른 계열": "OTHER",
    "바다가 떠오르는": "SCENE", "샤워하고 나온 것 같은": "SCENE",
    "살같에서 날 것 같은": "SOURCE",
    "담배냄새랑 섞여도 이상하지 않은": "PERFORMANCE",
    "땀냄새랑 섞여서 이상해지지 않는": "PERFORMANCE",
    "땀 냄새랑 섞여도 안 이상한 거": "PERFORMANCE",
    "땀이랑 안 겹치고": "PERFORMANCE", "체향에 잘 맞는": "PERFORMANCE",
    "슥 지나갔을 때": "PERFORMANCE", "무난하게 사용": "PERFORMANCE",
    "미니멀하게 옷 입는 사람": "CONTEXT", "스트릿룩이랑 어울리는": "CONTEXT",
    "밀폐된 공간": "CONTEXT", "사람 많은 곳": "CONTEXT",
    "올리브영이나 백화점": "CONTEXT", "면세점에서 사기 좋은": "CONTEXT",
    "디자인이 엄청 고급스럽고 화려해서 선물용으로 소장 가치 있는 향수 병": "OTHER",
}

def classify_annotation(field, expression):
    if field == "gold_avoid": return "AVOID"
    if expression in ROLE_OVERRIDES: return ROLE_OVERRIDES[expression]
    if has_any(expression, NEGATIVE_PATTERNS): return "AVOID"
    if has_any(expression, SCENE_PATTERNS): return "SCENE"
    if has_any(expression, REFERENCE_PATTERNS): return "REFERENCE"
    if has_any(expression, DIRECT_PATTERNS): return "DIRECT_SCENT"
    if has_any(expression, SOURCE_PATTERNS): return "SOURCE"
    if has_any(expression, PERFORMANCE_PATTERNS): return "PERFORMANCE"
    if has_any(expression, CONTEXT_PATTERNS): return "CONTEXT"
    if has_any(expression, IMAGE_PATTERNS): return "IMAGE"
    if has_any(expression, SENSORY_PATTERNS): return "SENSORY"
    return "OTHER"

occurrence_rows = []

def add_occurrence(query_id, query_text, expression, expression_type, extraction_origin):
    expression = expression.strip()
    if expression not in query_text:
        raise ValueError(f"원문 보존 실패: {query_id} / {expression!r}")
    occurrence_rows.append({
        "query_id": query_id, "query_text": query_text, "expression": expression,
        "expression_type": expression_type, "extraction_origin": extraction_origin,
    })

for row in gold_df.itertuples(index=False):
    for field in ["gold_avoid", "gold_additional_requirements"]:
        for item in getattr(row, field):
            expression = item.strip()
            add_occurrence(
                row.query_id, row.query_text, expression,
                classify_annotation(field, expression), field,
            )

SCENT_SURFACE_RULES = [
    ("DIRECT_SCENT", r"화이트 머스크"), ("DIRECT_SCENT", r"오렌지 블로썸"),
    ("DIRECT_SCENT", r"그린 노트"), ("DIRECT_SCENT", r"커피 노트"),
    ("DIRECT_SCENT", r"무화과 노트"), ("DIRECT_SCENT", r"시트러스(?:\s*계열)?"),
    ("DIRECT_SCENT", r"플로럴(?:\s*계열)?"), ("DIRECT_SCENT", r"아로마(?:틱)?\s*계열"),
    ("DIRECT_SCENT", r"그린\s*계열"), ("DIRECT_SCENT", r"시프레\s*계열"),
    ("DIRECT_SCENT", r"파우더리"), ("DIRECT_SCENT", r"우디"),
    ("DIRECT_SCENT", r"머스크"), ("DIRECT_SCENT", r"인센스"),
    ("DIRECT_SCENT", r"패출리"), ("DIRECT_SCENT", r"아이리스"),
    ("DIRECT_SCENT", r"베티버"), ("DIRECT_SCENT", r"스모키함"),
    ("DIRECT_SCENT", r"가죽 향"), ("DIRECT_SCENT", r"허브(?:\s*향)?"),
    ("SOURCE", r"레몬 껍질 냄새"), ("SOURCE", r"바다 냄새"),
    ("SOURCE", r"나무 냄새"), ("SOURCE", r"코코넛 향기"),
    ("SOURCE", r"섬유유연제 향"), ("SOURCE", r"비누\s*향"),
    ("SOURCE", r"홍차향"), ("SOURCE", r"베이비 파우더"),
    ("SOURCE", r"과일냄새"), ("SOURCE", r"꿀 향"),
    ("SOURCE", r"장미 꽃 향기"), ("SOURCE", r"자스민 냄새"),
    ("SOURCE", r"카라멜 향기"), ("SOURCE", r"딸기 향기"),
    ("SOURCE", r"오래된 헌책방 냄새"), ("SOURCE", r"살냄새"),
    ("SOURCE", r"세제 냄새"), ("SOURCE", r"절 냄새"),
    ("SOURCE", r"꽃향"), ("SOURCE", r"바닐라(?:\s*향기)?"),
    ("SOURCE", r"코코넛"), ("SOURCE", r"복숭아"), ("SOURCE", r"딸기"),
    ("SOURCE", r"장미"), ("SOURCE", r"자스민"), ("SOURCE", r"사과"),
    ("SOURCE", r"바질"), ("SOURCE", r"카라멜"), ("SOURCE", r"초코"),
    ("SOURCE", r"체리"), ("SOURCE", r"커피"), ("SOURCE", r"무화과"),
]
CONTEXT_SURFACE_RULES = [
    ("CONTEXT", r"\d+대(?:\s*(?:초반|후반))?"), ("CONTEXT", r"\d+살"),
    ("CONTEXT", r"사계절"), ("CONTEXT", r"봄"), ("CONTEXT", r"여름"),
    ("CONTEXT", r"가을"), ("CONTEXT", r"겨울"), ("CONTEXT", r"남성"),
    ("CONTEXT", r"여성(?!스러운)"), ("CONTEXT", r"남자(?!\s*향수\s*냄새)"),
    ("CONTEXT", r"여자(?!스러운)"),
]
IMAGE_SURFACE_RULES = [
    ("IMAGE", r"여성스러운"), ("IMAGE", r"중성적인"), ("IMAGE", r"중성적"),
    ("IMAGE", r"어른스러운"), ("IMAGE", r"세련된"), ("IMAGE", r"차분한"),
    ("IMAGE", r"섹시한"),
]
PERFORMANCE_SURFACES = {
    "UQ0030": ["너무 강하게 느껴지지 않는", "은은한"],
    "UQ0034": ["향수 뿌린 티 안나는"], "UQ0048": ["강하게 나는"],
    "UQ0056": ["너무 진한 향", "은은한"], "UQ0065": ["은은하게", "잔향 좋은"],
    "UQ0066": ["은은한"], "UQ0091": ["은은한"], "UQ0092": ["향이 너무 세지 않은"],
    "UQ0106": ["지속력 좋은"], "UQ0117": ["오래가는"],
    "UQ0138": ["향이 오래가는"], "UQ0139": ["은은한"],
    "UQ0147": ["지속력은 길었으면", "향이 멀리까지 퍼지는 건 싫어"],
    "UQ0163": ["너무 냄새가 강하지는 않았으면"], "UQ0178": ["지속력 좋은"],
    "UQ0180": ["잔향 오래가는"], "UQ0197": ["민폐 아닌 강도의", "가벼운"],
}
FIXED_SUPPLEMENTS = {
    "UQ0009": [
        ("너무 여성스럽지 않되", "AVOID"),
        ("너무 남자 향수도 아니었으면", "AVOID"),
    ],
}

def occupied_spans(query_id, query_text):
    spans = []
    for row in occurrence_rows:
        if row["query_id"] == query_id:
            start = query_text.find(row["expression"])
            spans.append((start, start + len(row["expression"])))
    return spans

def overlaps(start, end, spans):
    return any(start < old_end and old_start < end for old_start, old_end in spans)

for row in gold_df.itertuples(index=False):
    for expression, role in FIXED_SUPPLEMENTS.get(row.query_id, []):
        add_occurrence(row.query_id, row.query_text, expression, role, "fixed_surface_supplement")

    spans = occupied_spans(row.query_id, row.query_text)
    for role, pattern in SCENT_SURFACE_RULES + IMAGE_SURFACE_RULES + CONTEXT_SURFACE_RULES:
        for match in re.finditer(pattern, row.query_text, flags=re.IGNORECASE):
            if overlaps(match.start(), match.end(), spans):
                continue
            add_occurrence(row.query_id, row.query_text, match.group(0), role, "fixed_surface_pattern")
            spans.append((match.start(), match.end()))

    for expression in PERFORMANCE_SURFACES.get(row.query_id, []):
        start = row.query_text.find(expression)
        if start < 0:
            raise ValueError(f"performance 원문 불일치: {row.query_id} / {expression}")
        if overlaps(start, start + len(expression), spans):
            continue
        role = "AVOID" if "싫어" in expression or "않았으면" in expression else "PERFORMANCE"
        add_occurrence(row.query_id, row.query_text, expression, role, "gold_performance_surface")
        spans.append((start, start + len(expression)))

occurrence_df = (
    pd.DataFrame(occurrence_rows)
    .drop_duplicates(subset=["query_id", "expression", "expression_type"])
    .reset_index(drop=True)
)
if not set(occurrence_df["expression_type"]).issubset(ROLES):
    raise ValueError("허용되지 않은 expression_type")
if not all(
    expression in query
    for expression, query in occurrence_df[["expression", "query_text"]].itertuples(index=False, name=None)
):
    raise ValueError("원문에 없는 표현이 있습니다.")

print("Expression occurrences:", len(occurrence_df))
print("Queries with at least one expression:", occurrence_df["query_id"].nunique(), "/ 200")
display(occurrence_df.head(12))

Expression occurrences: 407
Queries with at least one expression: 200 / 200


,query_id,query_text,expression,expression_type,extraction_origin
0,UQ0001,비가 온 뒤의 숲같은 향을 찾고 있어,비가 온 뒤,SCENE,gold_additional_requirements
1,UQ0001,비가 온 뒤의 숲같은 향을 찾고 있어,숲,SCENE,gold_additional_requirements
2,UQ0002,시트러스 계열의 깔끔한 향의 향수가 있어?,깔끔한,SENSORY,gold_additional_requirements
3,UQ0003,향수 가격 적당한 거 추천해줘. 그냥 구색 갖추려고 사는 거야.,가격 적당,OTHER,gold_additional_requirements
4,UQ0003,향수 가격 적당한 거 추천해줘. 그냥 구색 갖추려고 사는 거야.,구색 갖추려고,OTHER,gold_additional_requirements
5,UQ0004,"여성스러운 향을 사고 싶은데, 너무 달콤한 건 싫어. 그런 향기가 있을까?",달콤한,AVOID,gold_avoid
6,UQ0005,난 바닐라 향기가 좋은데. 너무 단 건 싫어.,너무 단,AVOID,gold_avoid
7,UQ0007,30대 초반 남자가 쓰기 좋은 향수 추천해줘,30대 초반,CONTEXT,gold_additional_requirements
8,UQ0010,"케이크 같으면서도, 너무 느끼하지 않고, 딱 맡았을 때 공주처럼 느껴지는 향수 그런...",너무 느끼,AVOID,gold_avoid
9,UQ0010,"케이크 같으면서도, 너무 느끼하지 않고, 딱 맡았을 때 공주처럼 느껴지는 향수 그런...",공주,IMAGE,gold_additional_requirements


## 4. 원천 관계 판정과 v0.1 생성

관계 판정은 표면 문자열에 한정한다. `RELATED`도 의미상 동의어나 향조 관계가 아니라,
원천 표현의 포함 관계 또는 한국어 굴절형 공유만 뜻한다.

In [5]:
def normalize_lookup(value):
    value = unicodedata.normalize("NFKC", str(value)).casefold()
    return re.sub(r"[^0-9a-z가-힣]+", "", value)

STEM_STOPWORDS = {
    "향", "향수", "냄새", "느낌", "같", "좋", "추천", "사용", "쓰", "나",
    "너무", "약간", "정도", "사람", "있는", "없", "것", "거", "나는",
    "같은", "좋은", "머금은", "다른", "남자", "여자", "남성", "여성",
    "봄", "여름", "가을", "겨울", "은근히", "뿌리기", "많은", "가장",
    "향이", "냄새가", "선호", "반응",
}
STEM_SUFFIXES = [
    "스럽지않았으면", "하지않았으면", "하지않은", "하지않고", "하지", "했으면",
    "하면서", "하지만", "한데", "하게", "하고", "해서", "한", "함",
    "스럽", "로운", "롭게", "러운데", "러워", "러운", "거나", "지는", "은", "는", "운", "고", "게", "지",
    "가", "이", "을", "를", "에", "도", "와", "과", "랑", "로",
]

def lexical_stems(value):
    stems = set()
    for token in re.findall(r"[가-힣]+", unicodedata.normalize("NFKC", str(value))):
        stem = token
        for suffix in STEM_SUFFIXES:
            if len(stem) - len(suffix) >= 2 and stem.endswith(suffix):
                stem = stem[:-len(suffix)]
                break
        for source, replacement in [
            ("무거운", "무거"), ("무겁", "무거"), ("가벼운", "가벼"),
            ("가볍", "가벼"), ("부드러운", "부드러"), ("부드럽", "부드러"),
            ("날카로운", "날카로"), ("날카롭", "날카로"),
        ]:
            if stem.startswith(source):
                stem = stem.replace(source, replacement, 1)
        if len(stem) >= 2 and stem not in STEM_STOPWORDS:
            stems.add(stem)
    return stems

source_index = defaultdict(list)
for row in source_terms_df.itertuples(index=False):
    source_index[normalize_lookup(row.source_term)].append(row)
source_candidates = list(source_terms_df.itertuples(index=False))

FORCE_NOT_FOUND_EXPRESSIONS = {
    "저렴한", "반응 좋은", "사람 많은 곳", "향이 너무 세",
    "SNS에서 인기 많은", "인기 많은", "가장 핫", "선호 없음",
}
FORCE_RELATED_TERMS = {"호텔 침구": "깨끗한 침구"}

def related_result(source_term):
    rows = source_terms_df.loc[source_terms_df["source_term"].eq(source_term)]
    if rows.empty:
        raise ValueError(f"고정 원천 표현을 찾지 못했습니다: {source_term}")
    families = sorted(value for value in rows["source_family"].unique() if value)
    sections = sorted(rows["source_section"].unique())
    return {
        "representative_expression": source_term,
        "source_relation": "RELATED",
        "source_term": source_term,
        "source_families": "|".join(families),
        "source_sections": "|".join(sections),
        "evidence_note": "원천 표현과 명시 명사구 일부가 일치; 의미 동일성은 주장하지 않음",
    }

def select_source_match(expression):
    if expression in FORCE_NOT_FOUND_EXPRESSIONS:
        return {
            "representative_expression": expression, "source_relation": "NOT_FOUND",
            "source_term": "", "source_families": "", "source_sections": "",
            "evidence_note": "동형 표현의 문맥 오탐을 제외한 뒤 원천 근거를 찾지 못함",
        }
    if expression in FORCE_RELATED_TERMS:
        return related_result(FORCE_RELATED_TERMS[expression])
    normalized = normalize_lookup(expression)
    exact_rows = source_index.get(normalized, [])
    if exact_rows:
        representatives = [EXPLICIT_VARIANTS.get(row.source_term, row.source_term) for row in exact_rows]
        representative = sorted(representatives, key=lambda value: (len(value), value))[0]
        families = sorted({
            family for row in exact_rows for family in row.source_family.split("|") if family
        })
        sections = sorted({row.source_section for row in exact_rows})
        return {
            "representative_expression": representative,
            "source_relation": "MATCHED",
            "source_term": "|".join(sorted({row.source_term for row in exact_rows})),
            "source_families": "|".join(families),
            "source_sections": "|".join(sections),
            "evidence_note": "원천 문서 명시 표현/표기 변형과 정규화 일치",
        }

    expression_stems = lexical_stems(expression)
    related = []
    for row in source_candidates:
        term_norm = normalize_lookup(row.source_term)
        if len(term_norm) < 3:
            continue
        source_stems = lexical_stems(row.source_term)
        shared_stems = expression_stems & source_stems
        shorter = min(len(normalized), len(term_norm))
        longer = max(len(normalized), len(term_norm))
        containment = (
            shorter >= 3
            and (term_norm in normalized or normalized in term_norm)
            and shorter / longer >= 0.5
        )
        significant_shared = {stem for stem in shared_stems if len(stem) >= 2}
        stem_relation = bool(significant_shared) and (
            len(expression_stems) == 1 or len(source_stems) == 1
        )
        if containment or stem_relation:
            score = (
                2 if containment else 1, len(significant_shared),
                min(len(normalized), len(term_norm)), -abs(len(normalized) - len(term_norm)),
            )
            related.append((score, row))

    if related:
        best_score = max(score for score, _ in related)
        best_rows = [row for score, row in related if score == best_score]
        representative = sorted(
            {row.source_term for row in best_rows}, key=lambda value: (len(value), value)
        )[0]
        families = sorted({
            family for row in best_rows for family in row.source_family.split("|") if family
        })
        sections = sorted({row.source_section for row in best_rows})
        return {
            "representative_expression": representative,
            "source_relation": "RELATED",
            "source_term": "|".join(sorted({row.source_term for row in best_rows})),
            "source_families": "|".join(families),
            "source_sections": "|".join(sections),
            "evidence_note": "원천 표현과 부분 문자열 또는 굴절 어간 일치; 의미 동일성은 주장하지 않음",
        }

    return {
        "representative_expression": expression, "source_relation": "NOT_FOUND",
        "source_term": "", "source_families": "", "source_sections": "",
        "evidence_note": "원천 문서 명시 어휘에서 보수적 문자열/굴절 근거를 찾지 못함",
    }

aggregated_rows = []
for (expression, expression_type), group in occurrence_df.groupby(
    ["expression", "expression_type"], sort=False
):
    query_ids = sorted(group["query_id"].unique())
    example_query_id = query_ids[0]
    example_query_text = group.loc[group["query_id"].eq(example_query_id), "query_text"].iloc[0]
    aggregated_rows.append({
        "expression": expression, "expression_type": expression_type,
        "query_count": len(query_ids), "query_ids": "|".join(query_ids),
        "example_query_text": example_query_text, **select_source_match(expression),
    })

lexicon_df = pd.DataFrame(aggregated_rows)
lexicon_df["_role_order"] = lexicon_df["expression_type"].map(ROLE_ORDER)
lexicon_df = lexicon_df.sort_values(
    ["query_count", "_role_order", "expression"],
    ascending=[False, True, True], kind="stable",
).drop(columns="_role_order").reset_index(drop=True)

output_columns = [
    "expression", "representative_expression", "expression_type", "query_count",
    "query_ids", "example_query_text", "source_relation", "source_term",
    "source_families", "source_sections", "evidence_note",
]
lexicon_df = lexicon_df[output_columns]
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
lexicon_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Saved:", OUTPUT_PATH.relative_to(ROOT), "rows:", len(lexicon_df))
display(lexicon_df.head(15))

Saved: data\scent_knowledge\korean_scent_lexicon_v0_1.csv rows: 346


,expression,representative_expression,expression_type,query_count,query_ids,example_query_text,source_relation,source_term,source_families,source_sections,evidence_note
0,남자,남자,CONTEXT,22,UQ0007|UQ0023|UQ0041|UQ0047|UQ0085|UQ0089|UQ01...,30대 초반 남자가 쓰기 좋은 향수 추천해줘,NOT_FOUND,,,,원천 문서 명시 어휘에서 보수적 문자열/굴절 근거를 찾지 못함
1,우디,우디,DIRECT_SCENT,5,UQ0087|UQ0114|UQ0131|UQ0150|UQ0196,가을에 잘 어울리는 우디 향수 추천해줘,MATCHED,우디,Woods,0. 분류 기준 / Family 명칭|원천 문서 table line 1610,원천 문서 명시 표현/표기 변형과 정규화 일치
2,여름,여름,CONTEXT,5,UQ0015|UQ0016|UQ0042|UQ0085|UQ0132,여름에 뿌릴 향수 추천 좀.,NOT_FOUND,,,,원천 문서 명시 어휘에서 보수적 문자열/굴절 근거를 찾지 못함
3,중성적인,중성적인,IMAGE,4,UQ0009|UQ0067|UQ0071|UQ0140,"중성적인 향수 뿌리고 싶어. 너무 여성스럽지 않되, 너무 남자 향수도 아니었으면 좋겠다.",MATCHED,중성적인,,원천 문서 line 1685|원천 문서 table line 1613,원천 문서 명시 표현/표기 변형과 정규화 일치
4,여자,여자,CONTEXT,4,UQ0018|UQ0083|UQ0124|UQ0143,여자친구가 생겼는데 여친한테 점수 딸만한 향기 없나...?,NOT_FOUND,,,,원천 문서 명시 어휘에서 보수적 문자열/굴절 근거를 찾지 못함
5,은은한,은은한,PERFORMANCE,4,UQ0030|UQ0056|UQ0091|UQ0139,사람 많은 곳에서도 너무 강하게 느껴지지 않는 은은한 향수를 추천해줘.,MATCHED,은은한,Floral|Soft Floral,원천 문서 line 1668|원천 문서 line 1683|핵심 형용사,원천 문서 명시 표현/표기 변형과 정규화 일치
6,파우더리,파우더리한,DIRECT_SCENT,3,UQ0111|UQ0177|UQ0190,남자가 쓰기 좋은 파우더리 향수 추천해줘,RELATED,파우더리한,Floral Amber|Soft Amber|Soft Floral,바닐라/통카 쪽으로 갈 때|원천 문서 line 1107|핵심 형용사,원천 표현과 부분 문자열 또는 굴절 어간 일치; 의미 동일성은 주장하지 않음
7,깔끔한,깔끔한,SENSORY,3,UQ0002|UQ0023|UQ0028,시트러스 계열의 깔끔한 향의 향수가 있어?,MATCHED,깔끔한,Aromatic Fougère|Citrus|Woods,모던 푸제르 쪽|베르가못/자몽 계열|핵심 형용사,원천 문서 명시 표현/표기 변형과 정규화 일치
8,부드러운,부드러운,SENSORY,3,UQ0067|UQ0078|UQ0114,중성적인 향수 원해. 약간 담백하고 부드러운데 엣지있는거?,MATCHED,부드러운,Citrus|Floral|Floral Amber|Fruity|Soft Amber|S...,과육 표현|복숭아/살구 계열|샌달우드 쪽|오렌지/만다린 계열|핵심 형용사|화이트 플...,원천 문서 명시 표현/표기 변형과 정규화 일치
9,회사,회사,CONTEXT,3,UQ0028|UQ0075|UQ0091,회사에서 매일 사용해도 부담 없는 깔끔한 향수 추천해줘.,NOT_FOUND,,,,원천 문서 명시 어휘에서 보수적 문자열/굴절 근거를 찾지 못함


## 5. 검증과 요약

저장본을 다시 읽어 스키마, 역할, 관계 상태, Query ID, 원문 보존과 원본 입력 hash를 확인한다.
통계는 Lexicon v0.1의 기술 통계이며 전체 한국어 사용자 분포로 일반화하지 않는다.

In [6]:
saved_df = pd.read_csv(OUTPUT_PATH, dtype=str, keep_default_na=False, encoding="utf-8-sig")
if saved_df.columns.tolist() != output_columns or len(saved_df) != len(lexicon_df):
    raise ValueError("저장본 schema/row 불일치")
if saved_df.duplicated(["expression", "expression_type"]).any():
    raise ValueError("expression + expression_type 중복")
if not set(saved_df["expression_type"]).issubset(ROLES):
    raise ValueError("저장본 expression_type 오류")
if not set(saved_df["source_relation"]).issubset({"MATCHED", "RELATED", "NOT_FOUND"}):
    raise ValueError("저장본 source_relation 오류")

query_text_by_id = gold_df.set_index("query_id")["query_text"].to_dict()
for row in saved_df.itertuples(index=False):
    ids = row.query_ids.split("|")
    if int(row.query_count) != len(ids) or len(ids) != len(set(ids)):
        raise ValueError(f"query_count/query_ids 불일치: {row.expression}")
    if any(query_id not in query_text_by_id for query_id in ids):
        raise ValueError(f"알 수 없는 query_id: {row.expression}")
    if any(row.expression not in query_text_by_id[query_id] for query_id in ids):
        raise ValueError(f"원문 표현 보존 실패: {row.expression}")

input_hashes_after = {path: file_sha256(path) for path in required}
if input_hashes_after != input_hashes_before:
    raise RuntimeError("입력 Golden Set 또는 원천 문서가 변경되었습니다.")

type_summary = (
    saved_df.groupby("expression_type", observed=False).size()
    .reindex(ROLES, fill_value=0).rename("expression_count").reset_index()
)
unique_surface_count = saved_df["expression"].nunique()
relation_base = saved_df.drop_duplicates("expression")
relation_summary = (
    relation_base.groupby("source_relation", observed=False).size()
    .reindex(["MATCHED", "RELATED", "NOT_FOUND"], fill_value=0)
    .rename("expression_count").reset_index()
)
relation_summary["ratio"] = relation_summary["expression_count"] / unique_surface_count
saved_df["_query_count_int"] = saved_df["query_count"].astype(int)
frequent = saved_df.sort_values(
    ["_query_count_int", "expression"], ascending=[False, True]
).head(15)[["expression", "expression_type", "query_count", "source_relation"]]
repeated_not_found = saved_df.loc[
    saved_df["source_relation"].eq("NOT_FOUND") & saved_df["_query_count_int"].ge(2),
    ["expression", "expression_type", "query_count", "query_ids", "_query_count_int"],
].sort_values(["_query_count_int", "expression"], ascending=[False, True]).drop(
    columns="_query_count_int"
)

print("Unique surface expressions:", unique_surface_count)
print("Expression-role rows:", len(saved_df))
print("Covered queries:", occurrence_df["query_id"].nunique(), "/ 200")
print("\nType counts")
display(type_summary)
print("\nSource relation")
display(relation_summary.style.format({"ratio": "{:.1%}"}))
print("\nFrequent expressions by query_count")
display(frequent)
print("\nRepeated NOT_FOUND expressions")
display(repeated_not_found)

print("\nInterpretation boundary")
print("- Accord Mapping 검토 후보군: DIRECT_SCENT, SOURCE, 일부 SENSORY/SCENE (별도 근거 검토 필요)")
print("- 직접 Accord 연결 금지군: IMAGE, CONTEXT, PERFORMANCE, REFERENCE, OTHER")
print("- AVOID는 긍정 target으로 직접 연결하지 않고, 향 표적을 검토하더라도 부정 polarity를 보존해야 함")
print("- 이 Notebook은 위 후보에 실제 Accord/Note target을 부여하지 않았습니다.")
print("\nValidation: PASS")

Unique surface expressions: 341
Expression-role rows: 346
Covered queries: 200 / 200

Type counts


,expression_type,expression_count
0,DIRECT_SCENT,22
1,SENSORY,35
2,SCENE,25
3,SOURCE,29
4,IMAGE,29
5,CONTEXT,58
6,PERFORMANCE,23
7,AVOID,50
8,REFERENCE,15
9,OTHER,60



Source relation


,source_relation,expression_count,ratio
0,MATCHED,39,11.4%
1,RELATED,104,30.5%
2,NOT_FOUND,198,58.1%



Frequent expressions by query_count


,expression,expression_type,query_count,source_relation
0,남자,CONTEXT,22,NOT_FOUND
2,여름,CONTEXT,5,NOT_FOUND
1,우디,DIRECT_SCENT,5,MATCHED
4,여자,CONTEXT,4,NOT_FOUND
5,은은한,PERFORMANCE,4,MATCHED
3,중성적인,IMAGE,4,MATCHED
7,깔끔한,SENSORY,3,MATCHED
8,부드러운,SENSORY,3,MATCHED
6,파우더리,DIRECT_SCENT,3,RELATED
9,회사,CONTEXT,3,NOT_FOUND



Repeated NOT_FOUND expressions


,expression,expression_type,query_count,query_ids
0,남자,CONTEXT,22,UQ0007|UQ0023|UQ0041|UQ0047|UQ0085|UQ0089|UQ01...
2,여름,CONTEXT,5,UQ0015|UQ0016|UQ0042|UQ0085|UQ0132
4,여자,CONTEXT,4,UQ0018|UQ0083|UQ0124|UQ0143
9,회사,CONTEXT,3,UQ0028|UQ0075|UQ0091
15,30대,CONTEXT,2,UQ0074|UQ0178
16,겨울,CONTEXT,2,UQ0136|UQ0182
22,괜찮은 향수,OTHER,2,UQ0043|UQ0050
23,반응 좋은,OTHER,2,UQ0045|UQ0090
17,봄,CONTEXT,2,UQ0086|UQ0101
18,소개팅,CONTEXT,2,UQ0025|UQ0089



Interpretation boundary
- Accord Mapping 검토 후보군: DIRECT_SCENT, SOURCE, 일부 SENSORY/SCENE (별도 근거 검토 필요)
- 직접 Accord 연결 금지군: IMAGE, CONTEXT, PERFORMANCE, REFERENCE, OTHER
- AVOID는 긍정 target으로 직접 연결하지 않고, 향 표적을 검토하더라도 부정 polarity를 보존해야 함
- 이 Notebook은 위 후보에 실제 Accord/Note target을 부여하지 않았습니다.

Validation: PASS
